# Noncommutativity grid analysis

**Purpose.** This notebook runs a systematic empirical sweep to measure how
much output post-processing transformations change the variance-based (Sobol)
sensitivity profile of a model.  It implements the scoring metrics from the
companion paper:

> Hettinger, D. (2026). *Non-Commutativity of Sobol Sensitivity Indices Under
> Output Transformations of Functional, Spatial, and Scalar Models.*
> Prepared for the *Electronic Journal of Statistics.*
> Source: `docs/manuscripts/noncommutativity_detailed.tex`.

The scientific question is: if you apply a transformation $\phi$ to a model
output before computing Sobol indices, do you get the same result as computing
Sobol indices first and then summarising?  Mathematically, does
$S_i(\phi \circ f) = S_i(f)$?  The answer is **no** for any nonlinear
$\phi$ (Proposition 1).  This notebook quantifies *how large* the discrepancy
is across a registry of benchmarks and transformations.

**What this notebook produces.**
Running all cells creates four CSV files in
`outputs/noncommutativity_grid_analysis/` (or the directory set by
`SABENCH_GRID_OUTPUT_DIR`):

| File | Contents |
|------|----------|
| `pair_status.csv` | One row per (benchmark, transform) pair; status, metadata, finite-ness checks |
| `noncommutativity_metrics.csv` | Computed metrics for pairs that passed all checks |
| `summary_by_transform.csv` | Mean and max of key metrics, grouped by transform |
| `summary_by_benchmark.csv` | Mean and max of key metrics, grouped by benchmark |

**How to run.**
```bash
# Fast smoke run (default — CI-safe, small N)
jupyter notebook notebooks/noncommutativity_grid_analysis.ipynb

# Full run (all pairs, larger sample)
SABENCH_GRID_N_BASE=1024 jupyter notebook notebooks/noncommutativity_grid_analysis.ipynb

# Tiny debug run (first 3 benchmarks, first 10 transforms)
SABENCH_GRID_MAX_BENCHMARKS=3 SABENCH_GRID_MAX_TRANSFORMS=10 \
  jupyter notebook notebooks/noncommutativity_grid_analysis.ipynb
```

**Dependencies.**  All scientific logic is in tested `sabench` package modules;
this notebook only orchestrates and exports results.  No notebook-local
scientific computations.

## Background: Sobol sensitivity analysis

Variance-based (Sobol) global sensitivity analysis decomposes the total output
variance of a model $Y = f(\mathbf{X})$ — with independent, uniformly
distributed inputs $X_1, \ldots, X_d$ — according to the contribution of each
input and each interaction.

**First-order Sobol index** for input $X_i$:

$$S_i = \frac{D_i}{\mathrm{Var}[Y]}, \qquad
  D_i = \mathrm{Var}_{X_i}\!\bigl[\mathbb{E}[Y \mid X_i]\bigr].$$

$S_i$ measures the fraction of total output variance explained by $X_i$ alone,
averaging over all other inputs.

**Total-effect Sobol index**:

$$S_i^T = 1 - \frac{\mathrm{Var}_{\mathbf{X}_{\sim i}}\!\bigl[\mathbb{E}[Y \mid \mathbf{X}_{\sim i}]\bigr]}{\mathrm{Var}[Y]},$$

where $\mathbf{X}_{\sim i}$ is all inputs except $X_i$.  $S_i^T$ measures the
total contribution of $X_i$, including all interactions.

**Interpretation.** If $S_i^T < \tau$ for some threshold $\tau$ (e.g.,
$\tau = 0.05$), input $X_i$ can be fixed at any value without materially
reducing output variance — the *factor-fixing* decision.  Whether that decision
changes when the output is transformed is the central question of this notebook.

**Estimation.** This notebook uses the Jansen (1999) estimator with the
Saltelli (2002) input design, requiring $N(d+2)$ model evaluations for $d$
inputs and $N$ base samples.  Estimated indices are clipped to $[0, 1]$ to
handle finite-sample noise.  For benchmarks with multi-dimensional outputs
(e.g., spatial grids, temporal trajectories), the per-input index is
aggregated as a variance-weighted mean across output dimensions:

$$\hat{S}_i = \frac{\sum_j \mathrm{Var}[Y_j] \cdot \hat{S}_{ij}}{\sum_j \mathrm{Var}[Y_j]},$$

clipped to $[0, 1]$.  This gives a single sensitivity *profile*
$(\hat{S}_1, \ldots, \hat{S}_d)$ per pair regardless of output dimensionality.

## The commutativity theorem and its consequences

**Proposition 1 (Commutativity iff affine).** A continuous nonconstant
pointwise transform $\phi : \mathbb{R} \to \mathbb{R}$ satisfies
$S_i(\phi \circ f) = S_i(f)$ for every admissible square-integrable model $f$
if and only if $\phi$ is affine (i.e., $\phi(y) = ay + b$, $a \neq 0$).

*Consequence for practitioners:* **every nonlinear transformation of model
output can change the Sobol sensitivity profile.**  Log transforms, power laws,
exceedance probabilities, block averages, cumulative sums of nonlinear functions
— none of these preserve Sobol indices in general.  The magnitude of the change
is empirically large for environmental and engineering transforms
($\bar{D} \approx 0.37$, $\bar{\Delta} \approx 0.60$) and smaller but
nonzero for temporal/linear operators.

**Proposition 3 (Scalar invariance under monotone transforms).** Strictly
monotone scalar transforms produce identical Decision Score $D$ on any scalar
benchmark.  The Sensitivity Shift $\Delta$ may still be nonzero.  This result
applies to scalar-output models only; spatial and temporal benchmarks are not
covered.  *Practical implication:* rank transforms and log transforms of
scalar outputs do not change which inputs are classified as active or inactive
by the factor-fixing threshold — but they do change the quantitative index
values.

**Change-of-support (Proposition 2).** The change-of-support (COS) operator
that block-averages a spatial output commutes with Sobol index computation if
and only if the spatial sensitivity structure is homogeneous.  Heterogeneous
spatial benchmarks (Campbell3D) show $13\times$ amplified COS noncommutativity
relative to homogeneous ones.

*All three propositions are proved in the companion paper and in its appendix
for total-effect indices.*

## Metric definitions

### Decision Score $D$

The Decision Score measures the classification impact of the transformation on
the factor-fixing decision at threshold $\tau$ (default $\tau = 0.05$).

Define the soft activity indicator using a sigmoid centred at $\tau$:

$$\sigma(s; \tau) = \frac{1}{1 + \exp\!\left(-\frac{s - \tau}{\tau}\right)}.$$

The Decision Score is then:

$$D = \frac{1}{d} \sum_{i=1}^{d} \bigl| \sigma(\hat{S}_i(Z); \tau) \;-\; \sigma(\hat{S}_i(Y); \tau) \bigr|,$$

where $Y$ is the raw benchmark output and $Z = \phi(Y)$ is the transformed
output.

- $D = 0$: the transformation had **no** effect on factor-fixing classification
  (all inputs cross the threshold the same way before and after).
- $D = 1$: maximal classification reversal for every input.
- $D \in [0, 1]$ and is directly comparable across benchmarks and transforms.
- The sigmoid width equals $\tau$, calibrating sensitivity around the threshold
  boundary.  Unlike a rank correlation, $D$ responds to absolute index changes
  even when rankings are preserved.

### Sensitivity Shift $\Delta$ (Bray-Curtis)

The Sensitivity Shift measures the raw redistribution of sensitivity mass,
using the Bray-Curtis dissimilarity:

$$\Delta = \frac{\sum_{i=1}^{d} |\hat{S}_i(Z) - \hat{S}_i(Y)|}{\sum_{i=1}^{d} \hat{S}_i(Z) + \sum_{i=1}^{d} \hat{S}_i(Y)}.$$

- $\Delta = 0$: the transform left the profile entirely intact.
- $\Delta = 1$: the transform completely replaced one sensitivity profile with
  a non-overlapping one.
- The pooled denominator prevents instability when $\hat{S}_i(Y) \approx 0$.
- Bray-Curtis dissimilarity is the standard measure of community composition
  change in ecology, repurposed here for sensitivity profiles.

### Auxiliary metrics

| Column | Definition |
|--------|-----------|
| `threshold_flip_s1` / `threshold_flip_st` | Number of inputs that crossed the hard threshold $\tau$ in either direction; integer count |
| `topk_changed_s1` / `topk_changed_st` | Boolean: whether the unordered top-$k$ most-influential input set changed |
| `max_abs_shift_s1` / `max_abs_shift_st` | $\max_i |\hat{S}_i(Z) - \hat{S}_i(Y)|$ |
| `mean_abs_shift_s1` / `mean_abs_shift_st` | $\frac{1}{d} \sum_i |\hat{S}_i(Z) - \hat{S}_i(Y)|$ |
| `spearman_s1` / `spearman_st` | Spearman rank correlation between raw and transformed profiles |
| `top_driver_y_s1` / `top_driver_y_st` | 0-based index of the top-ranked input in the **raw** profile |
| `top_driver_z_s1` / `top_driver_z_st` | 0-based index of the top-ranked input in the **transformed** profile |

*Both first-order (`_s1`) and total-effect (`_st`) metrics are reported.*

## Configuration

The configuration cell below sets all parameters for this run.  Override any
of these by setting the corresponding environment variable before launching
Jupyter, or by editing the cell directly.

| Parameter | Env var | Default | Meaning |
|-----------|---------|---------|---------|
| `N_BASE` | `SABENCH_GRID_N_BASE` | 128 | Base sample size for Saltelli design; actual evaluations = $N(d+2)$.  Use 512–2048 for publication-quality results. |
| `RNG_SEED` | `SABENCH_GRID_SEED` | 20260429 | Random seed for reproducibility.  Fixed across all pairs. |
| `TAU` | `SABENCH_GRID_TAU` | 0.05 | Factor-fixing threshold; 0.05 is the standard sensitivity-analysis convention (Saltelli et al., 2008). |
| `TOP_K` | `SABENCH_GRID_TOP_K` | 3 | Top-$k$ driver set size for `topk_changed` metric. |
| `MAX_BENCHMARKS` | `SABENCH_GRID_MAX_BENCHMARKS` | 0 (all) | Truncate benchmark list to first $n$ entries.  0 = use all. |
| `MAX_TRANSFORMS` | `SABENCH_GRID_MAX_TRANSFORMS` | 0 (all) | Truncate transform list to first $n$ entries.  0 = use all. |
| `OUTPUT_DIR` | `SABENCH_GRID_OUTPUT_DIR` | `outputs/noncommutativity_grid_analysis` | Directory for exported CSVs.  Created if absent. |

**CI and smoke runs.** The default `N_BASE=128` is intentionally small for fast
execution.  For a thorough run over the full registry ($\approx 30$ benchmarks
$\times$ $\approx 117$ transforms), use `N_BASE=512` or larger and allow
several minutes of wall time.

**Reproducibility.** The `RNG_SEED` is applied globally; all pairs use the same
Saltelli sample reuse design, making results fully reproducible from the seed.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd

from sabench.analysis.grid import evaluate_noncommutativity_grid
from sabench.benchmarks import BENCHMARK_REGISTRY
from sabench.transforms import TRANSFORM_REGISTRY

N_BASE = int(os.environ.get("SABENCH_GRID_N_BASE", "128"))
RNG_SEED = int(os.environ.get("SABENCH_GRID_SEED", "20260429"))
TAU = float(os.environ.get("SABENCH_GRID_TAU", "0.05"))
TOP_K = int(os.environ.get("SABENCH_GRID_TOP_K", "3"))
MAX_BENCHMARKS = int(os.environ.get("SABENCH_GRID_MAX_BENCHMARKS", "0"))
MAX_TRANSFORMS = int(os.environ.get("SABENCH_GRID_MAX_TRANSFORMS", "0"))

_HERE = Path.cwd()
_REPO_ROOT = _HERE.parent if _HERE.name == "notebooks" else _HERE
OUTPUT_DIR = Path(
    os.environ.get(
        "SABENCH_GRID_OUTPUT_DIR",
        str(_REPO_ROOT / "outputs" / "noncommutativity_grid_analysis"),
    )
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

benchmark_keys = tuple(BENCHMARK_REGISTRY)
transform_keys = tuple(TRANSFORM_REGISTRY)
if MAX_BENCHMARKS > 0:
    benchmark_keys = benchmark_keys[:MAX_BENCHMARKS]
if MAX_TRANSFORMS > 0:
    transform_keys = transform_keys[:MAX_TRANSFORMS]

print(
    f"Grid: {len(benchmark_keys)} benchmarks × {len(transform_keys)} transforms, "
    f"N_BASE={N_BASE}, TAU={TAU}, seed={RNG_SEED}."
)

## Execute the registry-driven grid

The `evaluate_noncommutativity_grid()` function evaluates every
(benchmark, transform) pair and returns a list of result objects.  Each result
captures:

1. A **compatibility check** — whether the transform supports the benchmark's
   output kind (scalar, spatial, temporal).  Incompatible pairs are retained
   as `skipped_incompatible` rows.
2. A **benchmark evaluation** — `N(d+2)` calls to `benchmark.evaluate()` using
   the Saltelli sampling design.
3. **Sobol index estimation** — Jansen (1999) estimator for both first-order
   $\hat{S}_i$ and total-effect $\hat{S}_i^T$, clipped to $[0, 1]$.
4. For multi-output benchmarks, **variance-weighted profile aggregation**
   produces one $d$-vector per index type.
5. **Metric computation** — $D$, $\Delta$, flip counts, rank correlation, etc.
6. Pair-level exceptions are caught and recorded as `failed` rows, not crashes.

After the cell runs, `df` contains one row per pair.  The `pair_status` column
records the coarse outcome; the `metrics_status` column records whether metrics
were computed.

**Typical `pair_status` values:**
- `computed` — all steps succeeded; metric columns are populated
- `skipped_incompatible` — transform does not support benchmark's output kind
- `failed` — an unexpected error occurred (inspect `reason` column)

**Typical `metrics_status` values:**
- `computed` — metrics are present
- `skipped_incompatible` — same reason as `pair_status`
- `failed_zero_variance` — transformed output had zero variance; $D$ and
  $\Delta$ are undefined
- `failed` — metric computation raised an error

In [ ]:
results = evaluate_noncommutativity_grid(
    benchmark_keys=benchmark_keys,
    transform_keys=transform_keys,
    n_base=N_BASE,
    seed=RNG_SEED,
    tau=TAU,
    top_k=TOP_K,
)
rows = [result.as_dict() for result in results]
df = pd.DataFrame(rows)

# Normalize column names to short forms used in exports and summaries
_RENAME = {
    "decision_score_s1": "D_s1",
    "decision_score_st": "D_st",
    "sensitivity_shift_s1": "delta_s1",
    "sensitivity_shift_st": "delta_st",
    "threshold_flip_count_s1": "threshold_flip_s1",
    "threshold_flip_count_st": "threshold_flip_st",
    "top_source_index_s1": "top_driver_y_s1",
    "top_source_index_st": "top_driver_y_st",
    "top_transformed_index_s1": "top_driver_z_s1",
    "top_transformed_index_st": "top_driver_z_st",
}
df = df.rename(columns={k: v for k, v in _RENAME.items() if k in df.columns})

print(f"Evaluated {len(df)} candidate pairs")
print(df["metrics_status"].value_counts().to_string())

## Exported tables and column glossary

### `pair_status.csv`

One row per (benchmark, transform) pair, regardless of outcome.

| Column | Description |
|--------|-------------|
| `benchmark_key` | Registry key, e.g. `"Ishigami"` |
| `transform_key` | Registry key, e.g. `"log_shift_pos"` |
| `pair_status` | Coarse outcome: `computed`, `skipped_incompatible`, `failed` |
| `metrics_status` | Fine-grained outcome of metric computation |
| `reason` | Human-readable reason string for non-`computed` rows |
| `benchmark_output_kind` | `scalar`, `spatial`, or `temporal` |
| `transform_mechanism` | `pointwise`, `spatial`, `temporal`, etc. |
| `transform_tags` | Semicolon-separated transform tags from registry |
| `n_base` | $N$ used in the Saltelli design |
| `seed` | Random seed |
| `n_inputs` | $d$, number of model inputs |
| `n_evaluations` | Total model calls $= N(d+2)$ |
| `raw_output_shape` | Shape of raw benchmark output matrix |
| `transformed_output_shape` | Shape after transformation |
| `raw_output_finite` | Whether all raw outputs were finite |
| `transformed_output_finite` | Whether all transformed outputs were finite |
| `raw_variance` | Total variance of raw output (scalar or aggregate) |
| `transformed_variance` | Total variance of transformed output |

### `noncommutativity_metrics.csv`

Only rows where `metrics_status == "computed"`.

| Column | Description |
|--------|-------------|
| `D_s1`, `D_st` | Decision Score for first-order and total-effect indices |
| `delta_s1`, `delta_st` | Bray-Curtis Sensitivity Shift |
| `threshold_flip_s1`, `threshold_flip_st` | Hard-threshold crossing count |
| `topk_changed_s1`, `topk_changed_st` | Whether top-$k$ driver set changed |
| `max_abs_shift_s1`, `max_abs_shift_st` | Maximum per-input absolute shift |
| `mean_abs_shift_s1`, `mean_abs_shift_st` | Mean per-input absolute shift |
| `spearman_s1`, `spearman_st` | Spearman rank correlation of profiles |
| `top_driver_y_s1`, `top_driver_y_st` | Top input index in **raw** profile |
| `top_driver_z_s1`, `top_driver_z_st` | Top input index in **transformed** profile |

### `summary_by_transform.csv` and `summary_by_benchmark.csv`

Aggregate statistics (mean and max) of the four key metrics grouped by
transform and by benchmark, respectively.  Useful for ranking which
transforms or benchmarks show the largest empirical noncommutativity.

In [ ]:
PAIR_STATUS_COLUMNS = [
    "benchmark_key",
    "transform_key",
    "pair_status",
    "metrics_status",
    "reason",
    "benchmark_output_kind",
    "transform_mechanism",
    "transform_tags",
    "n_base",
    "seed",
    "n_inputs",
    "n_evaluations",
    "raw_output_shape",
    "transformed_output_shape",
    "raw_output_finite",
    "transformed_output_finite",
    "raw_variance",
    "transformed_variance",
]
METRIC_COLUMNS = [
    "benchmark_key",
    "transform_key",
    "D_s1",
    "delta_s1",
    "D_st",
    "delta_st",
    "threshold_flip_s1",
    "threshold_flip_st",
    "topk_changed_s1",
    "topk_changed_st",
    "max_abs_shift_s1",
    "max_abs_shift_st",
    "mean_abs_shift_s1",
    "mean_abs_shift_st",
    "spearman_s1",
    "spearman_st",
    "top_driver_y_s1",
    "top_driver_z_s1",
    "top_driver_y_st",
    "top_driver_z_st",
]

df_status = df[[c for c in PAIR_STATUS_COLUMNS if c in df.columns]].copy()
df_metrics = df.loc[df["metrics_status"] == "computed", [c for c in METRIC_COLUMNS if c in df.columns]].reset_index(drop=True)

df_status.to_csv(OUTPUT_DIR / "pair_status.csv", index=False)
df_metrics.to_csv(OUTPUT_DIR / "noncommutativity_metrics.csv", index=False)

print(f"Wrote pair_status.csv ({len(df_status)} rows)")
print(f"Wrote noncommutativity_metrics.csv ({len(df_metrics)} rows)")
try:
    display(df_metrics.head(10)) if len(df_metrics) > 0 else print("No computed metric rows.")
except NameError:
    print(df_metrics.head(10).to_string()) if len(df_metrics) > 0 else print("No computed metric rows.")

## Summaries by transform and benchmark

These tables aggregate the four headline metrics —
$D_{s1}$, $\Delta_{s1}$, $D_{st}$, $\Delta_{st}$ — grouped by transform and
by benchmark.  Both mean and max are reported.

- **Mean** reflects the typical noncommutativity magnitude of a given
  transform across all benchmarks it ran on, or of a given benchmark across
  all transforms.
- **Max** identifies the most extreme single pair for that transform or
  benchmark.

High mean $D$ or $\Delta$ for a transform indicates it is systematically
disruptive to sensitivity rankings, regardless of the model.  High max with
low mean indicates a single benchmark is particularly sensitive to that
transform (e.g., near-threshold inputs that flip).

In [ ]:
SUMMARY_METRICS = ["D_s1", "delta_s1", "D_st", "delta_st"]

if not df_metrics.empty:
    available = [m for m in SUMMARY_METRICS if m in df_metrics.columns]

    df_by_transform = (
        df_metrics.groupby("transform_key")[available]
        .agg(["mean", "max"])
        .round(4)
    )
    df_by_transform.columns = [f"{agg}_{m}" for m, agg in df_by_transform.columns]
    df_by_transform = df_by_transform.reset_index()

    df_by_benchmark = (
        df_metrics.groupby("benchmark_key")[available]
        .agg(["mean", "max"])
        .round(4)
    )
    df_by_benchmark.columns = [f"{agg}_{m}" for m, agg in df_by_benchmark.columns]
    df_by_benchmark = df_by_benchmark.reset_index()

    df_by_transform.to_csv(OUTPUT_DIR / "summary_by_transform.csv", index=False)
    df_by_benchmark.to_csv(OUTPUT_DIR / "summary_by_benchmark.csv", index=False)

    print(f"Wrote summary_by_transform.csv ({len(df_by_transform)} rows)")
    print(f"Wrote summary_by_benchmark.csv ({len(df_by_benchmark)} rows)")
    print("\nTop 10 pairs by first-order sensitivity shift (delta_s1):")
    if "delta_s1" in df_metrics.columns:
        try:
            display(
                df_metrics.nlargest(10, "delta_s1")[
                    ["benchmark_key", "transform_key", "D_s1", "delta_s1", "D_st", "delta_st"]
                ].reset_index(drop=True)
            )
        except NameError:
            print(
                df_metrics.nlargest(10, "delta_s1")[
                    ["benchmark_key", "transform_key", "D_s1", "delta_s1", "D_st", "delta_st"]
                ].reset_index(drop=True).to_string()
            )
else:
    df_by_transform = pd.DataFrame(columns=["transform_key"])
    df_by_benchmark = pd.DataFrame(columns=["benchmark_key"])
    df_by_transform.to_csv(OUTPUT_DIR / "summary_by_transform.csv", index=False)
    df_by_benchmark.to_csv(OUTPUT_DIR / "summary_by_benchmark.csv", index=False)
    print("No computed pairs — summary tables written as empty.")

## Interpreting results

### What the scores mean

| Score range | $D$ (Decision Score) | $\Delta$ (Sensitivity Shift) |
|-------------|---------------------|------------------------------|
| $\approx 0$ | Transform has no effect on factor-fixing classification | Transform leaves sensitivity profile intact |
| $0.05$–$0.15$ | Small, possibly negligible classification perturbation | Modest redistribution of sensitivity mass |
| $0.15$–$0.30$ | Noticeable; some inputs cross the threshold | Material profile change |
| $> 0.30$ | Large; classification decisions would differ substantially | Dominant redistribution; profile is materially altered |

*These ranges are calibrated from empirical findings in the companion paper
and are indicative, not prescriptive.*

### Expected patterns from the companion paper

1. **Affine transforms** (e.g., `scale`, `shift`, `affine_linear`): $D = 0$
   exactly (to numerical precision).  This is the commutativity theorem.

2. **Strictly monotone scalar transforms** (e.g., `log_shift_pos`, `exp`,
   `sqrt_shift_pos`, `cbrt`, `arctan`, `tanh`): $D = 0$ on scalar benchmarks
   (Proposition 3).  $\Delta$ may be nonzero.  These transforms change the
   quantitative Sobol index values but do not flip any factor-fixing
   classifications.

3. **Environmental and engineering transforms** (e.g., flood thresholds, power
   laws, exceedance probabilities): mean $D \approx 0.36$–$0.37$ and mean
   $\Delta \approx 0.60$–$0.61$ in the companion paper.

4. **Temporal transforms** (cumulative sum, peak, exceedance duration): lower
   on average ($\bar{D} \approx 0.18$, $\bar{\Delta} \approx 0.39$).  Linear
   cumulative sum on the Damped Oscillator scores near zero, confirming the
   theorem.

5. **Spatial COS transforms** on Campbell3D: mean $D$ is $13\times$ higher than
   on the homogeneous Campbell2D benchmark.

### Sampling noise

With `N_BASE=128`, estimated Sobol indices have nontrivial sampling variance.
Small non-zero $D$ or $\Delta$ values (e.g., $< 0.01$) should not be
interpreted as evidence of noncommutativity — they may reflect estimator noise.
Increase `N_BASE` to 512 or 1024 to reduce noise before drawing conclusions
about near-zero metrics.

### When `metrics_status` is not `computed`

- `failed_zero_variance`: the transformation produced a constant output for
  this benchmark.  Check whether the transform's domain is compatible with
  the benchmark's output range.
- `skipped_incompatible`: the transform does not operate on the benchmark's
  output kind.  This is expected, not an error.
- `failed`: inspect the `reason` column.  Common causes are domain errors
  (e.g., log of a negative value) or unsupported output shapes.

In [ ]:
print("Pair status breakdown:")
print(df["pair_status"].value_counts().to_string())
print()
if not df_metrics.empty and "delta_s1" in df_metrics.columns:
    top = df_metrics.nlargest(1, "delta_s1").iloc[0]
    print(f"Largest first-order shift: {top['benchmark_key']} + {top['transform_key']}  delta_s1={top['delta_s1']:.4f}")
if not df_metrics.empty and "delta_st" in df_metrics.columns:
    top_st = df_metrics.nlargest(1, "delta_st").iloc[0]
    print(f"Largest total-effect shift: {top_st['benchmark_key']} + {top_st['transform_key']}  delta_st={top_st['delta_st']:.4f}")